<a href="https://colab.research.google.com/github/mayorPandachka/ml_course_housework_PN/blob/main/hw02/hw02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Загрузка данных
df = pd.read_csv('/content/Heart_disease_statlog.csv')  # укажите ваш путь к файлу

# Базовая информация
print("Размер датасета:", df.shape)
print()
print("\nПервые 5 строк:")
print(df.head())
print()
print("\nПоследние 5 строк:")
print(df.tail())
print()
print("\nИнформация о данных:")
print(df.info())
print()
print("\nОписательная статистика:")
print(df.describe())
print()
print("\nОписание строк:")
print(df['age'].describe())
print()

# Проверка пропусков
print("\nПропуски в данных:")
print(df.isnull().sum())
print()
print("\nПроцент пропусков:")
print((df.isnull().sum() / len(df)) * 100)
print()

#Выдаленне пустых радкоў і слупоў
df.dropna(axis = 0, how='all', inplace = True)
df.dropna(axis = 1, how='all', inplace = True)
#Выдаленне адсутнага ўзроста, рода і вынікаў ЭКГ (бо гэта ключавая інфармацыя для запаўнення іншых пустэчаў і наўпрост важная для дыягназа)
df.dropna(subset = ['age', 'sex', 'restekg'])

#Пры адсутнасці інфармацыі пра падвышаным цукры адбываецца запаўненне нормай
df['fbs'].fillna(0, inplace=True)

#Запаўненне наяўнасці сценакардыі пры нагрузках адпаведна модзе па ўзросце і родзе
if df['exang'].isnull().any():
        df['exang'] = df.groupby(['sex', pd.cut(df['age'], bins=[0, 50, 100])])['exang'].transform(
            lambda x: x.fillna(x.mode()[0] if len(x.mode()) > 0 else 0)
        )

#Запаўненне адсутнага ціску модай па ўзроставай групе
if df['trestbps'].isnull().any():
        df['trestbps'] = df.groupby('age_group')['trestbps'].transform(
            lambda x: x.fillna(x.mode())
        )

#Запаўненне халестэрына модай па ўзроставай групе і роду
if df['chol'].isnull().any():
        df['chol'] = df.groupby(['sex', pd.cut(df_filled['age'], bins=[0, 45, 55, 100])])['chol'].transform(
            lambda x: x.fillna(x.mode())
        )

#Запаўненне максімальнага пульса модай па ўзроставай групе і хваробе
if df['thalach'].isnull().any():
        df['thalach'] = df.groupby(['target', pd.cut(df['age'], bins=[0, 50, 100])])['thalach'].transform(
            lambda x: x.fillna(x.mode())
        )

if df['oldpeak'].isnull().any():
        df['oldpeak'] = df.groupby(['cp', 'restecg'])['oldpeak'].transform(
            lambda x: x.fillna(x.mode())
        )

#Запаўненне тыпа болю адпаведна модзе па дыягназу
if df['cp'].isnull().any():
        df['cp'] = df.groupby('target')['cp'].transform(
            lambda x: x.fillna(x.mode()[0] if len(x.mode()) > 0 else 0)
        )

#Нахіл МТ
if df['slope'].isnull().any():
        # Создадим правило на основе oldpeak
        def estimate_slope(oldpeak_value):
            if pd.isna(oldpeak_value):
                return 1  # flat как наиболее частый
            elif oldpeak_value < 1:
                return 0  # up sloping
            elif oldpeak_value < 2:
                return 1  # flat
            else:
                return 2  # down sloping

        mask = df['slope'].isna()
        df.loc[mask, 'slope'] = df.loc[mask, 'oldpeak'].apply(estimate_slope)


#Колькасць папсаваных сасудаў
if df['ca'].isnull().any():
        # Медицинская логика: чаще всего 0 сосудов поражено
        # Но лучше заполнять на основе диагноза
        df['ca'] = df.groupby('target')['ca'].transform(
            lambda x: x.fillna(x.median())
        )
        df['ca'].fillna(0, inplace=True)

# Таласамія - запаўняем на паставе іншых пркмет
if df['thal'].isnull().any():
        # Заполняем модой по группам
        df['thal'] = df.groupby(['target', 'cp'])['thal'].transform(
            lambda x: x.fillna(x.mode()[0] if len(x.mode()) > 0 else 2)
        )

# Проверка дубликатов
print("\nКоличество дубликатов:", df.duplicated().sum())
print()

# Проверка уникальных значений в категориальных признаках
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal', 'target']
for col in categorical_cols:
    print(f"\nУникальные значения в {col}:")
    print(df[col].value_counts().sort_index())
print()

# Проверка выбросов в числовых признаках
numerical_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"\n{col}: {len(outliers)} выбросов ({len(outliers)/len(df)*100:.2f}%)")


Размер датасета: (270, 14)


Первые 5 строк:
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   70    1   3       130   322    0        2      109      0      2.4      1   
1   67    0   2       115   564    0        2      160      0      1.6      1   
2   57    1   1       124   261    0        0      141      0      0.3      0   
3   64    1   3       128   263    0        0      105      1      0.2      1   
4   74    0   1       120   269    0        2      121      1      0.2      0   

   ca  thal  target  
0   3     1       1  
1   0     3       0  
2   0     3       1  
3   1     3       0  
4   1     1       0  


Последние 5 строк:
     age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  \
265   52    1   2       172   199    1        0      162      0      0.5   
266   44    1   1       120   263    0        0      173      0      0.0   
267   56    0   1       140   294    0        2      153      0      1.3   
268   57   

In [ ]:
from google.colab import drive
drive.mount('/content/drive')